# 11 — Descriptivos avanzados del subcorpus de salud

Equivalente al notebook `13_ciencia_admin_descriptivos.ipynb` de Karen.

Integra los resultados de BERTopic (07), POS (08) y NER (06) para producir
un análisis descriptivo completo del subcorpus de salud.

**Entrada:** `salud_tweets_final.parquet`, `corpus_cleaned.parquet`,  
`general_ner.parquet`, `salud_ner.parquet`,  
`verbos_salud_stanza.parquet`, `adjetivos_salud_stanza.parquet`, `sustantivos_salud_stanza.parquet`  
**Salida:** `descriptivos_avanzados_salud.xlsx`, nubes de palabras, figuras

In [ ]:
# ============================================================
# CELL 0 — CONFIG
# ============================================================
from pathlib import Path

DATA_PROCESSED = Path(r'C:\Users\afpue\OneDrive\Documentos\GitHub\icare\kMetodo\resultadosPropios')

COLS_SUBCATS = [
    'Salud_Economia_salud', 'Salud_Estadisticas_sanitarias',
    'Salud_Higiene', 'Salud_Control_alimentos',
    'Salud_Epidemiologia', 'Salud_Higiene_ambiental',
    'Salud_Lucha_enfermedades', 'Salud_Politica_drogas',
    'Salud_Toxicomania', 'Salud_Salud_mujer',
    'Salud_Materno_infantil', 'Salud_Mental',
]

print('[CONFIG] OK')


In [ ]:
# ============================================================
# CELL 1 — IMPORTS Y CARGA
# ============================================================
import ast, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import spacy
from wordcloud import WordCloud
from collections import Counter
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Corpus completo
corpus = pd.read_parquet(DATA_PROCESSED / 'corpus_cleaned.parquet')
corpus['Fecha'] = pd.to_datetime(corpus['Fecha'], errors='coerce')

# Subcorpus de salud
tweets_salud = pd.read_parquet(DATA_PROCESSED / 'salud_tweets_final.parquet')

# Merge para obtener metadatos completos
df_salud = tweets_salud.merge(
    corpus[['id_doc', 'Author_Normalized', 'Entidad', 'Fecha', 'Texto_limpio']],
    on='id_doc', how='left'
)
df_salud['Fecha'] = pd.to_datetime(df_salud['Fecha'], errors='coerce')

print(f'Corpus completo   : {len(corpus):,} tweets')
print(f'Subcorpus salud   : {len(df_salud):,} tweets')
df_salud.head(2)


In [ ]:
# ============================================================
# CELL 2 — PORCENTAJE MENSUAL SALUD / CORPUS TOTAL
# (Replica directa de Karen — misma lógica)
# ============================================================

total_mensual = corpus.groupby(
    pd.Grouper(key='Fecha', freq='ME')
).size()

salud_mensual = (
    df_salud.groupby(pd.Grouper(key='Fecha', freq='ME'))['id_doc']
    .nunique()
)

mensual = pd.DataFrame({'total': total_mensual, 'salud': salud_mensual}).fillna(0)
mensual['porcentaje'] = (mensual['salud'] / mensual['total'] * 100).round(2)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(mensual.index, mensual['porcentaje'],
        marker='o', linestyle='-', color='steelblue', linewidth=2)
ax.set_title('Porcentaje mensual de tweets de salud en el corpus (2019–2021)')
ax.set_ylabel('% de tweets')
ax.set_xlabel('Mes')
ax.grid(alpha=0.3)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=45, ha='right')

for x, y in zip(mensual.index, mensual['porcentaje']):
    if not np.isnan(y) and y > 0:
        ax.text(x, y + 0.3, f'{y:.1f}%', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'fig_porcentaje_mensual_salud_avanzado.png', dpi=300)
plt.show()
print('[GUARDADO] fig_porcentaje_mensual_salud_avanzado.png')


In [ ]:
# ============================================================
# CELL 3 — TOP AUTORES EN SUBCORPUS DE SALUD
# (Replica de Karen)
# ============================================================
top_autores = df_salud['Author_Normalized'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6))
top_autores.plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Top 10 autores con más tweets de salud')
ax.set_xlabel('Número de tweets')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'fig_top_autores_salud_avanzado.png', dpi=300)
plt.show()
print('[GUARDADO] fig_top_autores_salud_avanzado.png')


In [ ]:
# ============================================================
# CELL 4 — NUBE DE PALABRAS DEL SUBCORPUS DE SALUD
# (Replica de Karen)
# ============================================================
nlp = spacy.load("es_core_news_sm")
STOP_WORDS = nlp.Defaults.stop_words

# Stopwords extra de Twitter
extra_sw = {'rt', 'http', 'https', 'co', 'amp', 'q', 'x', 'xq', 'via'}
STOP_WORDS = STOP_WORDS | extra_sw

texto_salud = ' '.join(df_salud['Texto_limpio'].dropna().tolist()).lower()

wc = WordCloud(
    width=1100, height=800,
    background_color='white',
    max_words=200,
    stopwords=STOP_WORDS,
    prefer_horizontal=1.0,
    relative_scaling=0.9,
    collocations=False,
    random_state=42,
    min_font_size=10,
    colormap='Blues',
    normalize_plurals=False
).generate(texto_salud)

plt.figure(figsize=(12, 8))
plt.imshow(wc, interpolation='bilinear')
plt.title('Nube de palabras — subcorpus de salud', fontsize=15, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'nube_subcorpus_salud_avanzado.png', dpi=300)
plt.show()
print('[GUARDADO] nube_subcorpus_salud_avanzado.png')


In [ ]:
# ============================================================
# CELL 5 — ANÁLISIS NER: EXTRAER Y TABULAR ENTIDADES
# (Replica de Karen — adaptada a la estructura de general_ner)
# ============================================================

def extract_entities(entities_array):
    """Extrae entidades del formato numpy array del NER."""
    entities_list = []
    if entities_array is None:
        return entities_list
    try:
        for item in entities_array:
            if hasattr(item, '__len__') and len(item) >= 2:
                entities_list.append((str(item[0]), str(item[1])))
    except Exception:
        pass
    return entities_list


# Cargar NER completo
ner_general = pd.read_parquet(DATA_PROCESSED / 'general_ner.parquet')

# Claves del subcorpus de salud
salud_ids = set(df_salud['id_doc'])

# Filtrar NER al subcorpus de salud
ner_salud = ner_general[ner_general['id_doc'].isin(salud_ids)].copy()
print(f'Tweets en general_ner    : {len(ner_general):,}')
print(f'Tweets de salud en NER   : {len(ner_salud):,}')


In [ ]:
# ============================================================
# CELL 6 — TABLA DE ENTIDADES (salud vs. corpus completo)
# (Replica directa de Karen)
# ============================================================

entidades_con_etiqueta = []

for _, row in tqdm(ner_general.iterrows(), total=len(ner_general), desc='NER global'):
    entidades = extract_entities(row['entidades'])
    es_salud  = 1 if row['id_doc'] in salud_ids else 0

    for tipo, nombre in entidades:
        entidades_con_etiqueta.append({
            'tipo'         : tipo,
            'nombre'       : nombre,
            'etiqueta_salud': es_salud,
        })

df_ent = pd.DataFrame(entidades_con_etiqueta)

resultado_ner = (
    df_ent.groupby(['tipo', 'nombre'])
    .agg(
        frecuencia_total  = ('etiqueta_salud', 'size'),
        frecuencia_salud  = ('etiqueta_salud', lambda x: (x == 1).sum()),
    )
    .reset_index()
)

resultado_ner['frecuencia_no_salud'] = (
    resultado_ner['frecuencia_total'] - resultado_ner['frecuencia_salud']
)
resultado_ner['cociente_salud'] = (
    resultado_ner['frecuencia_salud'] / resultado_ner['frecuencia_total']
).round(4)

resultado_ner.columns = [
    'Tipo', 'Entidad',
    'Frecuencia_Total', 'Frecuencia_Salud',
    'Frecuencia_No_Salud', 'Cociente_Salud'
]

resultado_ner = resultado_ner.sort_values(
    'Frecuencia_Total', ascending=False
).reset_index(drop=True)

print(f'Entidades únicas: {len(resultado_ner):,}')
print(resultado_ner.head(15).to_string(index=False))


In [ ]:
# ============================================================
# CELL 7 — ANÁLISIS POS: FRECUENCIAS EN SALUD vs. GENERAL
# (Replica de Karen — usa salidas del notebook 08)
# ============================================================
import ast

def asegurar_dict(x):
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return {}
    return {}


def construir_tabla_frecuencias_pos(df, columna_dicts, columna_salud='etiqueta_salud'):
    """
    Construye tabla: palabra, frecuencia_total, frecuencia_salud,
    proporcion_salud.
    """
    freq_total  = Counter()
    freq_salud  = Counter()

    for _, row in tqdm(df.iterrows(), total=len(df)):
        dic = asegurar_dict(row[columna_dicts])
        if not dic:
            continue
        freq_total.update(dic)
        if row[columna_salud] == 1:
            freq_salud.update(dic)

    data = []
    for palabra, f_total in freq_total.items():
        f_salud    = freq_salud.get(palabra, 0)
        prop_salud = f_salud / f_total if f_total > 0 else 0
        data.append({
            'palabra'            : palabra,
            'frecuencia_total'   : f_total,
            'frecuencia_salud'   : f_salud,
            'proporcion_salud'   : round(prop_salud, 4),
        })

    df_res = pd.DataFrame(data)
    if df_res.empty:
        return df_res
    return df_res.sort_values('frecuencia_total', ascending=False).reset_index(drop=True)


# Cargar POS (salidas notebook 08)
ruta_v = DATA_PROCESSED / 'verbos_salud_stanza.parquet'
ruta_a = DATA_PROCESSED / 'adjetivos_salud_stanza.parquet'
ruta_s = DATA_PROCESSED / 'sustantivos_salud_stanza.parquet'

if ruta_v.exists():
    df_verbos     = pd.read_parquet(ruta_v)
    df_adjetivos  = pd.read_parquet(ruta_a)
    df_sustantivos = pd.read_parquet(ruta_s)

    print('Construyendo tablas POS...')
    tabla_verbos     = construir_tabla_frecuencias_pos(df_verbos,     'verbos_lemas_frecuencias')
    tabla_adjetivos  = construir_tabla_frecuencias_pos(df_adjetivos,  'adjetivos_lemas_frecuencias')
    tabla_sustantivos= construir_tabla_frecuencias_pos(df_sustantivos,'sustantivos_lemas_frecuencias')

    print('Top 10 verbos:'     , tabla_verbos['palabra'].head(10).tolist())
    print('Top 10 adjetivos:'  , tabla_adjetivos['palabra'].head(10).tolist())
    print('Top 10 sustantivos:', tabla_sustantivos['palabra'].head(10).tolist())
    POS_OK = True
else:
    print('[AVISO] Archivos POS no encontrados. Ejecuta primero 08_extraer_frecuencias_POS.ipynb')
    POS_OK = False


In [ ]:
# ============================================================
# CELL 8 — GRÁFICOS POS TOP PALABRAS
# ============================================================
if POS_OK:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for ax, tabla, titulo in zip(
        axes,
        [tabla_verbos, tabla_sustantivos, tabla_adjetivos],
        ['Top 20 Verbos', 'Top 20 Sustantivos', 'Top 20 Adjetivos']
    ):
        top20 = tabla.head(20).sort_values('frecuencia_salud', ascending=True)
        ax.barh(top20['palabra'], top20['frecuencia_salud'], color='mediumseagreen')
        ax.set_title(titulo)
        ax.set_xlabel('Frecuencia en salud')
        ax.tick_params(axis='y', labelsize=8)

    plt.suptitle('Frecuencias POS en subcorpus de salud', fontsize=13)
    plt.tight_layout()
    plt.savefig(DATA_PROCESSED / 'fig_pos_top_palabras_salud.png', dpi=300)
    plt.show()
    print('[GUARDADO] fig_pos_top_palabras_salud.png')
else:
    print('[SKIP] Gráficos POS omitidos — ejecuta primero el notebook 08.')


In [ ]:
# ============================================================
# CELL 9 — GUARDAR TODO EN EXCEL
# (Replica de Karen)
# ============================================================
ruta_excel = DATA_PROCESSED / 'descriptivos_avanzados_salud.xlsx'

with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:
    mensual.reset_index().to_excel(writer, sheet_name='Porcentaje_mensual', index=False)
    top_autores.reset_index().rename(
        columns={'index': 'Autor', 'Author_Normalized': 'Tweets'}
    ).to_excel(writer, sheet_name='Top_autores', index=False)
    resultado_ner.to_excel(writer, sheet_name='Entidades_NER_salud', index=False)

    if POS_OK:
        tabla_verbos.to_excel(writer,      sheet_name='POS_verbos',     index=False)
        tabla_adjetivos.to_excel(writer,   sheet_name='POS_adjetivos',  index=False)
        tabla_sustantivos.to_excel(writer, sheet_name='POS_sustantivos',index=False)

print('[GUARDADO] descriptivos_avanzados_salud.xlsx')
print()
print('Notebook 11 completado.')
print('Pipeline completo.')
